# 🎬 Movie Recommendation System — Collaborative Filtering

**Approaches covered:** User-Based CF · Item-Based CF · SVD Matrix Factorization  
**Dataset:** MovieLens (ratings, movies, tags, links)

## 1. Dataset Loading & Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
ratings = pd.read_csv('ratings.csv')
movies  = pd.read_csv('movies.csv')
tags    = pd.read_csv('tags.csv')
links   = pd.read_csv('links.csv')

print('ratings shape:', ratings.shape)
print('movies shape :', movies.shape)
print('tags shape   :', tags.shape)
print('links shape  :', links.shape)

In [ ]:
print('=== ratings sample ===')
display(ratings.head())
print('=== movies sample ===')
display(movies.head())
print('=== tags sample ===')
display(tags.head())

In [ ]:
n_users  = ratings['userId'].nunique()
n_movies = ratings['movieId'].nunique()
n_ratings = len(ratings)
sparsity = 1 - n_ratings / (n_users * n_movies)

print(f'Unique users   : {n_users:,}')
print(f'Unique movies  : {n_movies:,}')
print(f'Total ratings  : {n_ratings:,}')
print(f'Rating range   : {ratings["rating"].min()} – {ratings["rating"].max()}')
print(f'Mean rating    : {ratings["rating"].mean():.3f}')
print(f'Matrix density : {(1 - sparsity)*100:.2f}%  (sparsity {sparsity*100:.2f}%)')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

ratings['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

ratings_per_user = ratings.groupby('userId').size()
axes[1].hist(ratings_per_user, bins=50, color='coral', edgecolor='white')
axes[1].set_title('Ratings per User')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Users')

ratings_per_movie = ratings.groupby('movieId').size()
axes[2].hist(ratings_per_movie, bins=50, color='mediumseagreen', edgecolor='white')
axes[2].set_title('Ratings per Movie')
axes[2].set_xlabel('Number of Ratings')
axes[2].set_ylabel('Movies')

plt.tight_layout()
plt.show()

## 2. Data Cleaning

In [ ]:
print('Missing values in ratings:', ratings.isnull().sum().to_dict())
print('Missing values in movies :', movies.isnull().sum().to_dict())

before = len(ratings)
ratings = ratings.drop_duplicates(subset=['userId', 'movieId'], keep='last')
print(f'\nDuplicates removed from ratings: {before - len(ratings)}')

movies = movies.drop_duplicates(subset=['movieId'], keep='first')

ratings = ratings.dropna(subset=['userId', 'movieId', 'rating'])
movies  = movies.dropna(subset=['movieId', 'title'])

ratings['rating'] = ratings['rating'].clip(0.5, 5.0)

MIN_USER_RATINGS  = 20
MIN_MOVIE_RATINGS = 10

user_counts  = ratings['userId'].value_counts()
movie_counts = ratings['movieId'].value_counts()

active_users  = user_counts[user_counts  >= MIN_USER_RATINGS].index
active_movies = movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index

ratings_clean = ratings[ratings['userId'].isin(active_users) & ratings['movieId'].isin(active_movies)].copy()

print(f'\nAfter quality filtering:')
print(f'  Users  : {ratings_clean["userId"].nunique():,}  (min {MIN_USER_RATINGS} ratings each)')
print(f'  Movies : {ratings_clean["movieId"].nunique():,}  (min {MIN_MOVIE_RATINGS} ratings each)')
print(f'  Rows   : {len(ratings_clean):,}')

## 3. User-Item Matrix

In [ ]:
user_movie_matrix = ratings_clean.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
)

print('User-Movie Matrix shape:', user_movie_matrix.shape)
print(f'Sparsity: {user_movie_matrix.isnull().sum().sum() / user_movie_matrix.size * 100:.1f}%')
display(user_movie_matrix.iloc[:5, :8])

In [ ]:
user_movie_filled = user_movie_matrix.fillna(0)

user_means = user_movie_matrix.mean(axis=1)
user_movie_normalized = user_movie_matrix.sub(user_means, axis=0).fillna(0)

print('Filled matrix (zeros for missing):', user_movie_filled.shape)
print('Mean-centered matrix:', user_movie_normalized.shape)

## 4. Similarity Calculation

In [ ]:
user_similarity_matrix = cosine_similarity(user_movie_normalized)
user_similarity_df = pd.DataFrame(
    user_similarity_matrix,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

item_similarity_matrix = cosine_similarity(user_movie_normalized.T)
item_similarity_df = pd.DataFrame(
    item_similarity_matrix,
    index=user_movie_matrix.columns,
    columns=user_movie_matrix.columns
)

print('User similarity matrix :', user_similarity_df.shape)
print('Item similarity matrix :', item_similarity_df.shape)

sample_users = user_similarity_df.iloc[:5, :5]
print('\nUser similarity sample (first 5 users):')
display(sample_users.round(3))

## 5. Recommendation Functions

In [ ]:
movie_id_to_title = movies.set_index('movieId')['title'].to_dict()

def recommend_movies_user_based(user_id, n=10, n_neighbors=20):
    if user_id not in user_similarity_df.index:
        return [f'User {user_id} not found in the dataset.']
    
    similar_users = (
        user_similarity_df[user_id]
        .drop(index=user_id)
        .nlargest(n_neighbors)
    )
    
    rated_by_user = set(user_movie_matrix.loc[user_id].dropna().index)
    
    weighted_scores = {}
    sim_sum = {}
    
    for neighbor, sim in similar_users.items():
        neighbor_ratings = user_movie_matrix.loc[neighbor].dropna()
        unrated = neighbor_ratings[~neighbor_ratings.index.isin(rated_by_user)]
        for movie_id, rating in unrated.items():
            weighted_scores[movie_id] = weighted_scores.get(movie_id, 0) + sim * rating
            sim_sum[movie_id] = sim_sum.get(movie_id, 0) + abs(sim)
    
    predicted = {m: weighted_scores[m] / sim_sum[m] for m in weighted_scores if sim_sum[m] > 0}
    top_movies = sorted(predicted, key=predicted.get, reverse=True)[:n]
    
    results = []
    for rank, movie_id in enumerate(top_movies, 1):
        title = movie_id_to_title.get(movie_id, f'Movie {movie_id}')
        score = predicted[movie_id]
        results.append(f'{rank:2}. {title}  (predicted rating: {score:.2f})')
    return results

In [ ]:
def recommend_movies_item_based(user_id, n=10, n_similar=20):
    if user_id not in user_movie_matrix.index:
        return [f'User {user_id} not found in the dataset.']
    
    user_ratings = user_movie_matrix.loc[user_id].dropna()
    rated_movies = set(user_ratings.index)
    
    predicted = {}
    
    for movie_id in item_similarity_df.index:
        if movie_id in rated_movies:
            continue
        if movie_id not in item_similarity_df.index:
            continue
        
        similar_items = (
            item_similarity_df[movie_id]
            .reindex(list(rated_movies))
            .dropna()
            .nlargest(n_similar)
        )
        
        if similar_items.empty:
            continue
        
        numerator   = sum(sim * user_ratings.get(m, 0) for m, sim in similar_items.items())
        denominator = similar_items.abs().sum()
        
        if denominator > 0:
            predicted[movie_id] = numerator / denominator
    
    top_movies = sorted(predicted, key=predicted.get, reverse=True)[:n]
    
    results = []
    for rank, movie_id in enumerate(top_movies, 1):
        title = movie_id_to_title.get(movie_id, f'Movie {movie_id}')
        score = predicted[movie_id]
        results.append(f'{rank:2}. {title}  (predicted rating: {score:.2f})')
    return results

In [ ]:
sample_user = user_movie_matrix.index[0]

print(f'=== User-Based Recommendations for User {sample_user} ===')
ub_recs = recommend_movies_user_based(sample_user)
for r in ub_recs:
    print(r)

print(f'\n=== Item-Based Recommendations for User {sample_user} ===')
ib_recs = recommend_movies_item_based(sample_user)
for r in ib_recs:
    print(r)

## 6. User-Based vs Item-Based Comparison

In [ ]:
test_users = user_movie_matrix.index[:5].tolist()

comparison_rows = []
for uid in test_users:
    ub = recommend_movies_user_based(uid, n=5)
    ib = recommend_movies_item_based(uid, n=5)
    
    ub_titles = set([r.split('.', 1)[1].split('(')[0].strip() for r in ub])
    ib_titles = set([r.split('.', 1)[1].split('(')[0].strip() for r in ib])
    overlap   = len(ub_titles & ib_titles)
    
    comparison_rows.append({'user_id': uid, 'ub_recs': len(ub), 'ib_recs': len(ib), 'overlap': overlap})

comp_df = pd.DataFrame(comparison_rows)
display(comp_df)

print("""
Quality Discussion
──────────────────
User-Based CF:
  ✔ Intuitive — finds taste neighbours and borrows their ratings.
  ✔ Discovers serendipitous titles outside a user's genre comfort zone.
  ✗ Scales poorly as O(users²); recomputing similarity is expensive.
  ✗ Suffers when users have very different rating scales (mitigated here
    with mean-centring before cosine similarity).

Item-Based CF:
  ✔ Item-item similarities are more stable over time than user-user ones.
  ✔ Can precompute and cache the item similarity matrix offline.
  ✔ Scales better for systems where #items ≪ #users.
  ✗ Tends to recommend within narrow genre clusters (less serendipity).
  ✗ Less effective for niche movies rated by very few users.
""")

## 7. Similar User Finder

In [ ]:
def find_similar_users(user_id, n=10):
    if user_id not in user_similarity_df.index:
        print(f'User {user_id} not found.')
        return None
    
    similar = (
        user_similarity_df[user_id]
        .drop(index=user_id)
        .nlargest(n)
        .reset_index()
    )
    similar.columns = ['similar_user_id', 'cosine_similarity']
    
    def common_ratings(other_id):
        u1 = user_movie_matrix.loc[user_id].dropna().index
        u2 = user_movie_matrix.loc[other_id].dropna().index
        return len(set(u1) & set(u2))
    
    similar['common_movies'] = similar['similar_user_id'].apply(common_ratings)
    return similar

print(f'=== Top 10 Users Similar to User {sample_user} ===')
similar_users_df = find_similar_users(sample_user)
display(similar_users_df)

## 8. Similar Movie Finder

In [ ]:
title_to_movie_id = (
    movies[movies['movieId'].isin(item_similarity_df.index)]
    .set_index('title')['movieId']
    .to_dict()
)

def similar_movies(movie_name, n=10):
    matches = [t for t in title_to_movie_id if movie_name.lower() in t.lower()]
    if not matches:
        print(f'No movie found matching "{movie_name}".')
        return None
    
    chosen_title = matches[0]
    movie_id = title_to_movie_id[chosen_title]
    print(f'Finding movies similar to: "{chosen_title}" (id={movie_id})')
    
    if movie_id not in item_similarity_df.index:
        print('Movie not in similarity matrix (too few ratings).')
        return None
    
    similar = (
        item_similarity_df[movie_id]
        .drop(index=movie_id)
        .nlargest(n)
        .reset_index()
    )
    similar.columns = ['movieId', 'cosine_similarity']
    similar['title'] = similar['movieId'].map(movie_id_to_title)
    similar['n_ratings'] = similar['movieId'].map(
        ratings_clean['movieId'].value_counts()
    )
    
    print(f'\nTop {n} movies similar to "{chosen_title}":')
    for i, row in similar.iterrows():
        print(f"  {i+1:2}. {row['title']}  (similarity: {row['cosine_similarity']:.3f}, ratings: {int(row['n_ratings']) if pd.notna(row['n_ratings']) else 'N/A'})")
    
    return similar

sim_df = similar_movies('Toy Story')

In [ ]:
sim_df2 = similar_movies('Matrix')

## 9. Visualization

In [ ]:
sample_size = 30
sample_users_idx = user_similarity_df.index[:sample_size]
user_sim_sample = user_similarity_df.loc[sample_users_idx, sample_users_idx]

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.eye(sample_size, dtype=bool)
sns.heatmap(
    user_sim_sample,
    mask=mask,
    annot=False,
    cmap='YlOrRd',
    vmin=0, vmax=1,
    linewidths=0.3,
    cbar_kws={'label': 'Cosine Similarity'},
    ax=ax
)
ax.set_title(f'User-User Similarity Heatmap (first {sample_size} users)', fontsize=14, pad=15)
ax.set_xlabel('User ID', fontsize=11)
ax.set_ylabel('User ID', fontsize=11)
ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
plt.show()

In [ ]:
top_movies_by_ratings = ratings_clean['movieId'].value_counts().head(25).index.tolist()
item_sim_sample = item_similarity_df.loc[top_movies_by_ratings, top_movies_by_ratings]

top_movie_labels = [movie_id_to_title.get(m, str(m))[:30] for m in top_movies_by_ratings]

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.eye(len(top_movies_by_ratings), dtype=bool)
sns.heatmap(
    item_sim_sample.values,
    mask=mask,
    annot=False,
    cmap='Blues',
    vmin=0, vmax=1,
    linewidths=0.4,
    xticklabels=top_movie_labels,
    yticklabels=top_movie_labels,
    cbar_kws={'label': 'Cosine Similarity'},
    ax=ax
)
ax.set_title('Item-Item Similarity Heatmap (Top 25 Most-Rated Movies)', fontsize=13, pad=15)
ax.tick_params(axis='x', rotation=45, labelsize=7)
ax.tick_params(axis='y', rotation=0, labelsize=7)
plt.tight_layout()
plt.show()

In [ ]:
svd_2d = TruncatedSVD(n_components=2, random_state=42)
user_vectors_2d = svd_2d.fit_transform(user_movie_filled.values)

n_clusters = 6
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(user_vectors_2d)

fig, ax = plt.subplots(figsize=(12, 8))
palette = sns.color_palette('husl', n_clusters)

for c in range(n_clusters):
    mask = cluster_labels == c
    ax.scatter(
        user_vectors_2d[mask, 0],
        user_vectors_2d[mask, 1],
        s=18,
        alpha=0.6,
        color=palette[c],
        label=f'Cluster {c+1}'
    )

ax.set_title('User Clusters via SVD (2D) + K-Means', fontsize=14)
ax.set_xlabel('SVD Component 1')
ax.set_ylabel('SVD Component 2')
ax.legend(title='Cluster', loc='best', fontsize=9)
plt.tight_layout()
plt.show()

cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
print('Cluster sizes:')
for c, size in cluster_sizes.items():
    print(f'  Cluster {c+1}: {size} users')

In [ ]:
top_n_movies = 100
top_movie_ids = ratings_clean['movieId'].value_counts().head(top_n_movies).index
item_vec = item_similarity_df.loc[top_movie_ids, top_movie_ids].values

svd_item = TruncatedSVD(n_components=2, random_state=42)
item_2d = svd_item.fit_transform(item_vec)

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(item_2d[:, 0], item_2d[:, 1], s=40, alpha=0.7, color='royalblue')

for i, mid in enumerate(top_movie_ids[:30]):
    title = movie_id_to_title.get(mid, str(mid))[:20]
    ax.annotate(title, (item_2d[i, 0], item_2d[i, 1]), fontsize=6.5, alpha=0.85)

ax.set_title(f'Movie Similarity Map (Top {top_n_movies} Movies, SVD 2D projection)', fontsize=13)
ax.set_xlabel('SVD Component 1')
ax.set_ylabel('SVD Component 2')
plt.tight_layout()
plt.show()

## 10. Matrix Factorization with SVD

In [ ]:
R = user_movie_filled.values.astype(np.float32)

user_ratings_mean = np.mean(R, axis=1).reshape(-1, 1)
R_demeaned = R - user_ratings_mean

N_FACTORS = 50
U, sigma, Vt = svds(csr_matrix(R_demeaned), k=N_FACTORS)

sigma_diag = np.diag(sigma)
R_predicted = np.dot(np.dot(U, sigma_diag), Vt) + user_ratings_mean
R_predicted = np.clip(R_predicted, 0.5, 5.0)

svd_predictions_df = pd.DataFrame(
    R_predicted,
    index=user_movie_filled.index,
    columns=user_movie_filled.columns
)

print(f'SVD matrix factorization complete.')
print(f'  Latent factors : {N_FACTORS}')
print(f'  U shape        : {U.shape}')
print(f'  sigma shape    : {sigma.shape}')
print(f'  Vt shape       : {Vt.shape}')
print(f'  R_predicted    : {R_predicted.shape}')

In [ ]:
svd_full = TruncatedSVD(n_components=min(100, min(R_demeaned.shape) - 1), random_state=42)
svd_full.fit(R_demeaned)

explained_var = svd_full.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(explained_var)+1), explained_var, color='steelblue', alpha=0.8)
axes[0].set_title('Explained Variance per Component')
axes[0].set_xlabel('SVD Component')
axes[0].set_ylabel('Explained Variance Ratio')

axes[1].plot(range(1, len(cumulative_var)+1), cumulative_var, marker='o', markersize=3, color='coral')
axes[1].axhline(0.90, color='gray', linestyle='--', label='90% threshold')
axes[1].axhline(0.80, color='lightgray', linestyle='--', label='80% threshold')
axes[1].set_title('Cumulative Explained Variance')
axes[1].set_xlabel('Number of SVD Components')
axes[1].set_ylabel('Cumulative Variance Ratio')
axes[1].legend()

plt.tight_layout()
plt.show()

n_for_80 = np.searchsorted(cumulative_var, 0.80) + 1
n_for_90 = np.searchsorted(cumulative_var, 0.90) + 1
print(f'Components needed for 80% variance: {n_for_80}')
print(f'Components needed for 90% variance: {n_for_90}')

In [ ]:
def recommend_movies_svd(user_id, n=10):
    if user_id not in svd_predictions_df.index:
        return [f'User {user_id} not found.']
    
    rated_by_user = user_movie_matrix.loc[user_id].dropna().index
    predicted_ratings = svd_predictions_df.loc[user_id]
    unrated_predicted = predicted_ratings.drop(index=rated_by_user, errors='ignore')
    top_movies = unrated_predicted.nlargest(n)
    
    results = []
    for rank, (movie_id, score) in enumerate(top_movies.items(), 1):
        title = movie_id_to_title.get(movie_id, f'Movie {movie_id}')
        results.append(f'{rank:2}. {title}  (SVD predicted rating: {score:.2f})')
    return results

print(f'=== SVD Recommendations for User {sample_user} ===')
svd_recs = recommend_movies_svd(sample_user)
for r in svd_recs:
    print(r)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sorted_sigma = np.sort(sigma)[::-1]
ax.plot(range(1, len(sorted_sigma)+1), sorted_sigma, 'o-', markersize=4, color='mediumorchid')
ax.set_title('Singular Values (Sigma) — Information Captured per Latent Factor')
ax.set_xlabel('Latent Factor Index')
ax.set_ylabel('Singular Value')
ax.fill_between(range(1, len(sorted_sigma)+1), sorted_sigma, alpha=0.15, color='mediumorchid')
plt.tight_layout()
plt.show()

## 11. Evaluation — RMSE & MAE

In [ ]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(ratings_clean, test_size=0.2, random_state=42)

print(f'Train size: {len(train_data):,}')
print(f'Test size : {len(test_data):,}')

train_matrix = train_data.pivot_table(index='userId', columns='movieId', values='rating').fillna(0)
test_matrix  = test_data.pivot_table(index='userId', columns='movieId', values='rating')

train_users  = train_matrix.index
train_movies = train_matrix.columns

test_filtered = test_data[
    test_data['userId'].isin(train_users) &
    test_data['movieId'].isin(train_movies)
]

print(f'Evaluable test samples (users+movies in train): {len(test_filtered):,}')

In [ ]:
R_train = train_matrix.values.astype(np.float32)
train_means = np.mean(R_train, axis=1).reshape(-1, 1)
R_train_dm  = R_train - train_means

U_t, s_t, Vt_t = svds(csr_matrix(R_train_dm), k=50)
R_pred_train = np.dot(np.dot(U_t, np.diag(s_t)), Vt_t) + train_means
R_pred_train = np.clip(R_pred_train, 0.5, 5.0)

pred_df = pd.DataFrame(R_pred_train, index=train_matrix.index, columns=train_matrix.columns)

actuals, svd_preds = [], []
for _, row in test_filtered.iterrows():
    uid, mid, actual = row['userId'], row['movieId'], row['rating']
    if uid in pred_df.index and mid in pred_df.columns:
        actuals.append(actual)
        svd_preds.append(pred_df.loc[uid, mid])

actuals   = np.array(actuals)
svd_preds = np.array(svd_preds)

svd_rmse = np.sqrt(mean_squared_error(actuals, svd_preds))
svd_mae  = mean_absolute_error(actuals, svd_preds)

print(f'SVD Matrix Factorization (k=50 factors):')
print(f'  RMSE : {svd_rmse:.4f}')
print(f'  MAE  : {svd_mae:.4f}')
print(f'  Evaluated on {len(actuals):,} test samples')

In [ ]:
global_mean = train_data['rating'].mean()
baseline_preds = np.full(len(actuals), global_mean)

baseline_rmse = np.sqrt(mean_squared_error(actuals, baseline_preds))
baseline_mae  = mean_absolute_error(actuals, baseline_preds)

user_bias = train_data.groupby('userId')['rating'].mean() - global_mean
movie_bias = train_data.groupby('movieId')['rating'].mean() - global_mean

biased_preds = []
for _, row in test_filtered.iterrows():
    uid, mid = row['userId'], row['movieId']
    ub = user_bias.get(uid, 0)
    mb = movie_bias.get(mid, 0)
    pred = np.clip(global_mean + ub + mb, 0.5, 5.0)
    biased_preds.append(pred)

biased_preds = np.array(biased_preds[:len(actuals)])
biased_rmse = np.sqrt(mean_squared_error(actuals, biased_preds))
biased_mae  = mean_absolute_error(actuals, biased_preds)

metrics = pd.DataFrame({
    'Model': ['Global Mean Baseline', 'Bias-adjusted Baseline', 'SVD (k=50 factors)'],
    'RMSE':  [baseline_rmse, biased_rmse, svd_rmse],
    'MAE':   [baseline_mae,  biased_mae,  svd_mae]
})
metrics[['RMSE','MAE']] = metrics[['RMSE','MAE']].round(4)
print('Model Comparison:')
display(metrics)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colours = ['#e07b54', '#5e9bd6', '#6abf69']

axes[0].bar(metrics['Model'], metrics['RMSE'], color=colours)
axes[0].set_title('RMSE by Model (lower is better)')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(metrics['Model'], metrics['MAE'], color=colours)
axes[1].set_title('MAE by Model (lower is better)')
axes[1].set_ylabel('MAE')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
errors = actuals - svd_preds

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(errors, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero error')
axes[0].set_title('SVD Prediction Error Distribution')
axes[0].set_xlabel('Actual − Predicted')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].scatter(svd_preds[:2000], actuals[:2000], alpha=0.15, s=10, color='coral')
axes[1].plot([0.5, 5], [0.5, 5], 'k--', linewidth=1, label='Perfect prediction')
axes[1].set_title('Predicted vs Actual Ratings (SVD)')
axes[1].set_xlabel('Predicted Rating')
axes[1].set_ylabel('Actual Rating')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Error statistics:')
print(f'  Mean error : {errors.mean():.4f}  (bias)')
print(f'  Std error  : {errors.std():.4f}')
print(f'  % within ±0.5 star : {(np.abs(errors) <= 0.5).mean()*100:.1f}%')
print(f'  % within ±1.0 star : {(np.abs(errors) <= 1.0).mean()*100:.1f}%')

## 12. Cold Start Analysis

In [ ]:
rating_counts = ratings_clean.groupby('userId').size()

buckets = [0, 5, 10, 20, 50, 100, 200, 500, float('inf')]
labels  = ['1-5', '6-10', '11-20', '21-50', '51-100', '101-200', '201-500', '500+']

rating_buckets = pd.cut(rating_counts, bins=buckets, labels=labels)
bucket_counts  = rating_buckets.value_counts().reindex(labels)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(labels, bucket_counts.values, color=sns.color_palette('viridis', len(labels)), edgecolor='white')
ax.set_title('User Activity Distribution — Cold Start Perspective', fontsize=13)
ax.set_xlabel('Number of Ratings per User')
ax.set_ylabel('Number of Users')

for bar, val in zip(bars, bucket_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, str(val),
            ha='center', va='bottom', fontsize=9)

ax.axvline(1.5, color='red', linestyle='--', alpha=0.5, label='Cold start zone (< 10 ratings)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def recommend_cold_start_popularity(n=10, min_ratings=50):
    movie_stats = (
        ratings_clean
        .groupby('movieId')
        .agg(avg_rating=('rating', 'mean'), n_ratings=('rating', 'count'))
        .reset_index()
    )
    
    C = movie_stats['avg_rating'].mean()
    m = min_ratings
    
    movie_stats['bayesian_score'] = (
        (movie_stats['n_ratings'] / (movie_stats['n_ratings'] + m)) * movie_stats['avg_rating'] +
        (m / (movie_stats['n_ratings'] + m)) * C
    )
    
    top = movie_stats.nlargest(n, 'bayesian_score')
    top['title'] = top['movieId'].map(movie_id_to_title)
    return top[['title', 'avg_rating', 'n_ratings', 'bayesian_score']].reset_index(drop=True)

print('=== Cold Start Fallback: Top 10 by Bayesian Average Rating ===')
cold_start_recs = recommend_cold_start_popularity(n=10)
display(cold_start_recs.round(3))

In [ ]:
def recommend_cold_start_by_genre(genre, n=10, min_ratings=20):
    genre_movies = movies[movies['genres'].str.contains(genre, case=False, na=False)]['movieId']
    
    genre_ratings = (
        ratings_clean[ratings_clean['movieId'].isin(genre_movies)]
        .groupby('movieId')
        .agg(avg_rating=('rating','mean'), n_ratings=('rating','count'))
        .query(f'n_ratings >= {min_ratings}')
        .reset_index()
    )
    
    C = genre_ratings['avg_rating'].mean()
    m = min_ratings
    genre_ratings['bayesian_score'] = (
        (genre_ratings['n_ratings'] / (genre_ratings['n_ratings'] + m)) * genre_ratings['avg_rating'] +
        (m / (genre_ratings['n_ratings'] + m)) * C
    )
    
    top = genre_ratings.nlargest(n, 'bayesian_score')
    top['title'] = top['movieId'].map(movie_id_to_title)
    return top[['title','avg_rating','n_ratings','bayesian_score']].reset_index(drop=True)

print('=== Cold Start: Top Action Movies ===')
display(recommend_cold_start_by_genre('Action', n=8).round(3))

print('\n=== Cold Start: Top Drama Movies ===')
display(recommend_cold_start_by_genre('Drama', n=8).round(3))

In [ ]:
print("""
Cold Start Problem — Summary
═══════════════════════════════════════════════════════════════

Two flavours of cold start exist in collaborative filtering:

1. NEW USER COLD START
   A user with zero (or very few) ratings has no neighbourhood
   in user-based CF and no rated anchor movies for item-based CF.
   SVD cannot place them in latent space.

   Mitigation strategies implemented above:
   a) Popularity fallback — recommend globally popular movies
      using a Bayesian average (avoids bias toward blockbusters
      with huge but average review counts).
   b) Genre/demographic onboarding — ask the user a few genre
      preferences during sign-up and use those as a proxy.
   c) Hybrid model — blend content-based signals (genres, tags)
      from the movies dataset with CF predictions.

2. NEW ITEM COLD START
   A newly added movie has no ratings, so it can never appear
   in item-based recommendations and gets no column in the
   user-item matrix.

   Mitigation strategies:
   a) Content-based bootstrapping — use movie metadata (genres,
      cast, crew) to find movies with similar content profiles.
   b) Tags from the tags.csv file carry rich semantic signals
      that can substitute for user ratings in early life.
   c) Use IMDb/TMDb links (links.csv) to fetch external review
      signals until enough internal ratings accumulate.

Threshold for reliable CF (observed in this dataset):
  ≥ 20 ratings per user  → neighbourhood is stable enough
  ≥ 10 ratings per movie → item similarity is meaningful
  Below these thresholds → always fall back to the strategies above.
═══════════════════════════════════════════════════════════════
""")

## Summary & Quick-Use Guide

In [ ]:
print("Quick-use guide — call any of these functions:\n")
print("  recommend_movies_user_based(user_id, n=10)")
print("    → CF from taste neighbours.")
print()
print("  recommend_movies_item_based(user_id, n=10)")
print("    → CF from rated-item neighbourhoods.")
print()
print("  recommend_movies_svd(user_id, n=10)")
print("    → SVD matrix factorization (Netflix-style).")
print()
print("  find_similar_users(user_id, n=10)")
print("    → Returns DataFrame of most similar users.")
print()
print("  similar_movies('Toy Story', n=10)")
print("    → Returns movies most similar to the given title.")
print()
print("  recommend_cold_start_popularity(n=10)")
print("    → Best fallback for brand-new users.")
print()
print("  recommend_cold_start_by_genre('Action', n=10)")
print("    → Genre-seeded cold start after onboarding survey.")

print("\n" + "="*60)
print(f"Dataset: {ratings_clean['userId'].nunique():,} users · "
      f"{ratings_clean['movieId'].nunique():,} movies · "
      f"{len(ratings_clean):,} ratings")
print(f"SVD — RMSE: {svd_rmse:.4f}  MAE: {svd_mae:.4f}")
print("="*60)